<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.3
# *Google BigQuery and Gemini API*

## Introduction

The Google BigQuery UI provides access to Google's extensive collection of public data sets via an SQL-based query engine.

The BigQuery API provides programmatic access to the data sets.

The Google Gemini API provides programmatic access to Google's Generative AI models.

Each of these is explored in this lab.

## BigQuery Web UI

The Google BigQuery UI can be used to discover interesting data before writing Python code to access it. Then we can reproduce it in an API request so as to aggregate large amounts of data on Google's infrastructure before pulling the results into our application.

Work through the Quickstart at https://cloud.google.com/bigquery/docs/quickstarts/quickstart-web-ui.

You will need to set up a Google Cloud Platform account if you don't already have one. (This should not cost anything during the trial period unless you perform a large amount of querying. Afterwards, costs are based on actual resource usage, but most offerings have a free tier.)

## BigQuery API

- Open Google Cloud Console (https://console.cloud.google.com/home/) and select to create a project. A project is required to enable access to Google Cloud services such as BigQuery and Gemini.

- Check that the BigQuery API is enabled in your project by visiting https://console.cloud.google.com/apis/library/bigquery.googleapis.com.

### Authentication

Create a **service account** at https://console.cloud.google.com/iam-admin/serviceaccounts/create. A service account is used by an application to access Google Cloud Platform's services and has an associated email address (different from your own).


- Give the account an appropriate name, and under step 2 (Grant this service account access to project (optional)), choose "Owner" under the "Select a Role" dropdown.

- Ignore step 3 and click "Done".

Go to https://console.cloud.google.com/iam-admin/serviceaccounts to create a **service account key**. This will be downloaded to your computer so that you can connect to the BigQuery API via this Jupyter notebook.

- Select your recently created project.
- Click the email address of the service account.
- Click the Keys tab.
- Click the Add key drop-down menu, then select Create new key.
- Select JSON as the Key type and click Create.
- The keys will get saved to your computer.

Note the location and copy the file path (of the json file) to somewhere safe, for future reference.

See here for more information:

Service Account creation: https://cloud.google.com/iam/docs/service-accounts-create#creating (under Console)

Service Account key creation: https://cloud.google.com/iam/docs/keys-create-delete#iam-service-account-keys-create-console (under Console)


### Using the Python API

Google provides Python libraries for wrapping the Google APIs.

Installing the "google-cloud-bigquery", "google-cloud-storage", and "google-cloud-bigquery-storage" libraries should cover all the dependencies for the BigQuery section of this lab.

In [1]:
!pip install google-cloud-bigquery
!pip install google-cloud-storage
!pip install google-cloud-bigquery-storage

   ---------------------------------------- 0.0/4.8 MB ? eta -:--:--
   -------- ------------------------------- 1.0/4.8 MB 5.6 MB/s eta 0:00:01
   ------------------- -------------------- 2.4/4.8 MB 5.5 MB/s eta 0:00:01
   ---------------------------- ----------- 3.4/4.8 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------  4.7/4.8 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 4.8/4.8 MB 5.5 MB/s eta 0:00:00

   ----------------------------------------  0/12 [rsa]
   ----------------------------------------  0/12 [rsa]
   ----------------------------------------  0/12 [rsa]
  Attempting uninstall: protobuf
   ----------------------------------------  0/12 [rsa]
    Found existing installation: protobuf 5.29.3
   ----------------------------------------  0/12 [rsa]
   --- ------------------------------------  1/12 [protobuf]
    Uninstalling protobuf-5.29.3:
   --- ------------------------------------  1/12 [protobuf]
      Successfully uninstalled p

In [2]:
from google.cloud import bigquery
from google.cloud import storage
from google.cloud import bigquery_storage

Invoke a method of the `.Client` object that takes the path to your key files as a string argument:

In [3]:
key_path = 'vcu-lab-323-30865a14b8ab.json'

This should not throw an error if key retrieval / assignment worked:

In [4]:
storage_client = storage.Client.from_service_account_json(key_path)

*Nb. The `storage` object was used in the above example, but there are other objects of interest that have polymorphic `Client` members that are used similarly, such as `bigquery`, which is used below.*

Next, execute this:

In [7]:
client = bigquery.Client.from_service_account_json(key_path)

This client is associated with the default project (which was set or defaulted in the BigQuery UI):

In [8]:
client.project

'vcu-lab-323'

A BigQuery project contains datasets. Datasets contain tables. To get at the data in a table we need to create a reference that covers this hierarchy; in the `bigquery` library this looks like `project.dataset.table`.  

(Nb. Queries can be performed on projects and datasets, but most queries are performed on tables.)

To explore the public datasets we will start by reassigning our `client` variable using optional `project` parameter (set to `bigquery-public-data`):

In [9]:
#project = 'bigquery-public-data'
client = bigquery.Client.from_service_account_json(key_path, project = 'bigquery-public-data')
print(client.project)

bigquery-public-data


Here is how to get a list of the datasets in the current project:

In [10]:
datasets = list(client.list_datasets())
print(datasets)

[<google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF34050>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEC65D0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEC6990>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEE29E0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEE22C0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF08CB0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF285A0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF286B0>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEE6450>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DEE6B50>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF5C050>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x000001910DF5C230>, <google.cloud.bigquery.dataset.DatasetListItem object at 0x0000

That wasn't helpful. We need to go deeper into the object structure to get at something meaningful. Below is a function that exploits the `format` method of `project` and `dataset_id`, providing an easy way to list datasets:

In [11]:
# function for listing datasets in a project:
def printDatasetList(client):
    project = client.project    #: only one project can be associated with a client instance
    datasets = list(client.list_datasets())
    if datasets:
        print('Datasets in project {}:'.format(project))
        for dataset in datasets:
            print('\t{}'.format(dataset.dataset_id))
        found = True
    else:
        print('{} project does not contain any datasets.'.format(project))
        found = False
    return found

In [12]:
# list datasets in the default project:
flag = printDatasetList(client)  #: assigning to `flag` suppresses printing the return value (normally `True`)

Datasets in project bigquery-public-data:
	agent_analytic_sample_dataset
	america_health_rankings
	austin_311
	austin_bikeshare
	austin_crime
	austin_incidents
	austin_waste
	baseball
	bbc_news
	bigqueryml_ncaa
	bitcoin_blockchain
	blackhole_database
	blockchain_analytics_ethereum_mainnet_us
	bls
	bls_qcew
	breathe
	broadstreet_adi
	catalonian_mobile_coverage
	catalonian_mobile_coverage_eu
	census_bureau_acs
	census_bureau_construction
	census_bureau_international
	census_bureau_usa
	census_opportunity_atlas
	census_utility
	cfpb_complaints
	chicago_crime
	chicago_taxi_trips
	clemson_dice
	cloud_storage_geo_index
	cms_codes
	cms_medicare
	cms_synthetic_patient_data_omop
	country_codes
	covid19_aha
	covid19_covidtracking
	covid19_ecdc
	covid19_ecdc_eu
	covid19_genome_sequence
	covid19_geotab_mobility_impact
	covid19_geotab_mobility_impact_eu
	covid19_google_mobility
	covid19_google_mobility_eu
	covid19_govt_response
	covid19_italy
	covid19_italy_eu
	covid19_jhu_csse
	covid19_jhu_csse_eu

This list should correspond to what is shown here https://bigquery.cloud.google.com/publicdatasets under the **bigquery-public-data** item.

Here is how to create a dataset reference object by assigning a project and a dataset name:

In [15]:
dataset_id = 'samples'
dataset_ref = client.dataset(dataset_id)

If our current project was something other than `bigquery-public-data`, we could still create this reference by specifying the project that contains the dataset:

In [16]:
dataset_id = 'samples'
dataset_ref = client.dataset(dataset_id, project = 'bigquery-public-data')

How can we get the path of the dataset?

In [17]:
#ANSWER:
print(dataset_ref.path)

/projects/bigquery-public-data/datasets/samples


Explore more of this object's members:

*(HINT: You can type `dataset_ref.` in a new line, then hit the [Tab] key to see the available members for the object.)*

In [19]:
#?
dataset_ref

DatasetReference('bigquery-public-data', 'samples')

Here is a function for listing the tables in a dataset:

In [20]:
# function for listing tables in a dataset:
def printTableList(client, dataset_id):
    project = client.project
    dataset_ref = client.dataset(dataset_id, project = project)
    tables = list(client.list_tables(dataset_ref))
    if tables:
        print('Tables in dataset {}:'.format(dataset_id))
        for table in tables:
            print('\t{}'.format(table.table_id))
        found = True
    else:
        print('{} dataset does not contain any tables.'.format(dataset_id))
        found = False
    return found

Use this function to list the tables in the current dataset:

In [28]:
#ANSWER
printTableList(client, dataset_id)

Tables in dataset samples:
	github_nested
	github_timeline
	gsod
	natality
	shakespeare
	trigrams
	wikipedia


True

To create a reference to a table within the dataset, we use the `table_id` attribute:

In [25]:
table_id = 'shakespeare'
table_ref = dataset_ref.table(table_id)

To access the data in the table itself, we use the `get_table()` method:

In [26]:
table = client.get_table(table_ref)  # API Request

NOTE: The contents of the table are not actually in our memory after this call! We are working with a Big Data platform, now, and we could easily end up pulling GBs or TBs of data by accident.

To minimize data bandwidth, memory consumption, and processing time, Big Data platforms employ ***lazy evaluation***. This means that no computation or data transfer actually takes place until we *realize* (use) the data. Even if we execute subsequent code that performs calculations on the data, no data flow or computation actually occurs until we request output (e.g. by executing a print to stdout or writing to a file).

What kind of object is returned by `client.get_table`?

In [29]:
#ANSWER:
type(table)

google.cloud.bigquery.table.Table

How can we view the design of the table (column names and types)? The name of the object attribute we need is the same term we learned in the module on databases:

In [30]:
#ANSWER
table.schema

[SchemaField('word', 'STRING', 'REQUIRED', None, 'A single unique word (where whitespace is the delimiter) extracted from a corpus.', (), None, None),
 SchemaField('word_count', 'INTEGER', 'REQUIRED', None, 'The number of times this word appears in this corpus.', (), None, None),
 SchemaField('corpus', 'STRING', 'REQUIRED', None, 'The work from which this word was extracted.', (), None, None),
 SchemaField('corpus_date', 'INTEGER', 'REQUIRED', None, 'The year in which this corpus was published.', (), None, None)]

Again, this is messy. If we wanted to refer to the column names and types in code, we might use something like this (which we could then parse into a dict):

In [31]:
result = ["{0} {1}".format(schema.name,schema.field_type) for schema in table.schema]
print(result)

['word STRING', 'word_count INTEGER', 'corpus STRING', 'corpus_date INTEGER']


But if we just want to print them, here is another neat function for that:

In [32]:
# function to print a table schema:
def printTableSchema(aTable):
    schemas = list(aTable.schema)
    if schemas:
        print('Table schema for {}:'.format(aTable.table_id))
        for aSchema in schemas:
            print('\t{0} {1}'.format(aSchema.name, aSchema.field_type))
        found = True
    else:
        found = False
    return found

Use this function to print the table schema:

In [33]:
#ANSWER:
printTableSchema(table)

Table schema for shakespeare:
	word STRING
	word_count INTEGER
	corpus STRING
	corpus_date INTEGER


True

Now that we know what the columns are, we can write queries. Actually, we construct a query job by assigning an SQL statement to a method of the `client` object:

In [34]:
sql = "SELECT COUNT(1) FROM bigquery-public-data.samples.shakespeare"
query_job = client.query(sql)

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/bigquery-public-data/jobs?prettyPrint=false: Access Denied: Project bigquery-public-data: User does not have bigquery.jobs.create permission in project bigquery-public-data.

Location: None
Job ID: cba35def-16fa-45e0-b538-d8349351b87b


This will throw an error since we don't have permission to create queries inside the `bigquery-public-data` project. Instead we set the project to our BigQuery project name.

In [37]:
client = bigquery.Client.from_service_account_json(key_path, project = 'vcu-lab-323') #<<< your BigQuery project ID here!
query_job = client.query(sql)

If that worked, show what query_job is:

In [38]:
# ANSWER
query_job

QueryJob<project=vcu-lab-323, location=US, id=09a92805-10a2-4fa9-9aba-46176c908f18>

Once again, due to lazy execution, no actual execution occurs until we request output:

In [39]:
for row in query_job:  # API request - fetches results
    print(row)

Row((164656,), {'f0_': 0})


And, again, we need to manipulate this to make it neat. Each member of the rowset is a list and we only want to extract the value, which is in the first member:

In [40]:
print(row[0])

164656


So, we now know that this table has 164,656 rows. (We would not want to print it!)

Write, execute, and print the results of a query that fetches 10 rows from the table, each containing the "word", "word_count", and "corpus" fields:

In [ ]:
#ANSWER
sql = "SELECT * FROM 

Whenever you catch yourself writing a swag of code to do something that seems rudimentary or low-level, there is a very good chance that you don't need to. A much easier way to handle the above requirement is to use the `to_dataframe` method of the QueryJob object:

In [ ]:
df = query_job.to_dataframe()
df

#### Additional Notes

1. Here is a readable way to code long SQL statements:

In [ ]:
sql = """
    SELECT word, word_count, corpus
    FROM bigquery-public-data.samples.shakespeare
    LIMIT 10
    """

2. If you had an application that needed to modify the tables or datasets in the `bigquery-public-data` is project, you could copy them to our own project, where you would have the permissions to do as you please with the data (subject to Google's terms of use).

3. We aren't limited to the datasets that are already in BigQuery. We can upload tables from our computer, and we can pull data in from other online sources.

## Google Gemini

Google Gemini (formerly Bard) is a multimodal generative AI chatbot. It can process text, audio, images and video.

Create an API key at https://aistudio.google.com/app/apikey . Copy the key and paste it into a text file called 'gemini_key.txt'

## Google Gemini UI
While signed into Google experiment with some prompts at https://aistudio.google.com/app/prompts/new_chat. A prompt gallery is available at https://aistudio.google.com/app/gallery.

## Google Gemini API

The library `google-generativeai` gives access to Gemini models. For this section download the following two files from the DATA folder:

* `equation.jpg`
* `JFK.mp3`

In [41]:
# -U gives the latest version
!pip install -U google-genai

   ---------------------------------------- 0.0/724.2 kB ? eta -:--:--
   ---------------------------------------- 724.2/724.2 kB 5.6 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
   ---------------------------------------- 0/3 [websockets]
  Attempting uninstall: anyio
   ---------------------------------------- 0/3 [websockets]
    Found existing installation: anyio 4.7.0
   ---------------------------------------- 0/3 [websockets]
   ------------- -------------------------- 1/3 [anyio]
    Uninstalling anyio-4.7.0:
   ------------- -------------------------- 1/3 [anyio]
      Successfully uninstalled anyio-4.7.0
   ------------- -------------------------- 1/3 [anyio]
   ------------- -------------------------- 

In [42]:
from google import genai
from IPython.display import Markdown # allows Markdown text to be displayed in the notebook

Firstly we read our API key from `gemini_key.txt`:

In [43]:
filename = 'gemini_key.txt'
try:
    with open(filename, 'r') as f:
        # It's assumed your file contains a single line containing your API key only
        key = f.read().strip()
except FileNotFoundError:
    print("'%s' file not found" % filename)

In [44]:
client = genai.Client(api_key=key)

We shall use the following client methods:

* `models.generate_content()`: used to generate responses from the model
* `models.list()`: used to see available models
* `files.upload()`: used to upload image/audio files

The following code lists the available models for text generation:

In [45]:
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


As suggested by the name, Gemini Flash is designed for faster responses while Gemini Pro works better at more challenging tasks. Pro has a lower rate limit of 2 requests per minute as seen in https://ai.google.dev/pricing.

In [46]:
response = client.models.generate_content(
    model="gemini-2.5-flash", contents="What is an API?"
)

The attribute `text` shows in Markdown format the response of the model to the question.

In [47]:
Markdown(response.text)

An **API** (Application Programming Interface) is a set of rules, protocols, and tools that allows different software applications to communicate and interact with each other.

Think of it as a **menu and a waiter** in a restaurant:

*   **You (the "client application")** want something done (e.g., get a weather forecast, send a tweet, make a payment).
*   **The Menu (the "API Documentation")** tells you what you can order (what services or data are available) and how to order it (what format your request should be in).
*   **The Waiter (the "API" itself)** takes your order (your request) to the kitchen (the "server application" or "backend system").
*   **The Kitchen (the "server application")** prepares your order (processes the request).
*   **The Waiter (the "API")** brings you back your food (the response or data you requested).

**Key aspects of an API:**

1.  **Interface/Contract:** It defines the methods and data formats that applications can use to request and exchange information. It acts as a contract, outlining what one system can ask for and what the other system will provide in return.
2.  **Abstraction:** APIs hide the complexity of the underlying system. When you use an API, you don't need to know *how* the other application works internally, only *how* to interact with it through its defined interface.
3.  **Communication:** APIs are the messengers that carry requests from one application to another and then deliver the responses back.
4.  **Standardization:** Many APIs use standard communication protocols like HTTP/HTTPS and data formats like JSON or XML, making it easier for different systems to understand each other.

**Common Examples of APIs in Action:**

*   **Social Media APIs:** When a third-party app lets you "Login with Facebook" or "Tweet this article," it's using Facebook's or Twitter's API.
*   **Weather APIs:** Weather apps on your phone or computer don't have their own weather stations; they get data from a weather service's API.
*   **Payment Gateways:** When you pay online using services like Stripe or PayPal, the e-commerce website uses their APIs to securely process your payment without ever touching your sensitive credit card information directly.
*   **Google Maps API:** Websites embed interactive maps by calling the Google Maps API, which handles all the map rendering and data.
*   **Operating System APIs:** When a program saves a file or displays a window, it's using the operating system's API to perform those actions.

**Why are APIs important?**

*   **Interoperability:** They enable different software systems, even those built with different technologies, to work together seamlessly.
*   **Faster Development:** Developers don't have to "reinvent the wheel." They can leverage existing functionalities and services from other platforms, speeding up development.
*   **Innovation:** APIs allow companies to open up their platforms, letting other developers build new and innovative services on top of them.
*   **Efficiency:** They create modular systems, making it easier to maintain and update components independently.

In essence, APIs are the **invisible connectors** that power much of the modern digital world, allowing countless applications and services to share information and functionality, creating a more integrated and dynamic user experience.

Study the response object and identify the total token count.

In [ ]:
# ANSWER


Next, we have Gemini process a mathematical equation in an image.

In [ ]:
sample_image = client.files.upload(file="equation.jpg")

In [ ]:
result = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        sample_image,
        "\n\n",
        "What is in this image?",
    ],
)
Markdown(result.text)

Use the Gemini 1.5 Flash model to solve the equation in the image.

In [ ]:
# REPLACE ??? with code
result = client.models.generate_content(
    model=???,
    contents=[
        sample_image,
        "\n\n",
        ???,
    ],
)

Markdown(">" + result.text)

Finally we transcribe a short audio clip.

In [ ]:
# REPLACE ??? with code
audio_file = client.files.upload(file="JFK.mp3")

# Create a prompt.
prompt = "Transcribe the audio."

# Pass the prompt and the audio file to gemini-2.5-flash using client.models.generate_content
response = ???

# Print the response.
print(response.text)

## Further reading

If you wish to pick up a few more skills in BigQuery you can go to https://cloud.google.com/bigquery/create-simple-app-api and https://cloud.google.com/bigquery/docs/samples.

Alternatively, you can take a deeper dive into the API here: https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/usage.html.

The Google Gemini API documentation is at https://ai.google.dev/gemini-api/docs

## - END -



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



